# Non-Python workers

> **Runtime compatibility:** the OpenSSL worker in this lesson uses the Rust subprocess executor and the standard worker protocol.

Tierkreis works easiest when running with python workers.
Still it supports using arbitrary workers that provide a binary to run.
In this example we will look at using a shell script as a worker.
Conceptually, this also works with other programming languages, for example if you want to use a legacy HPC application.

To make this work we need to introduce three changes:
1. (Optional) We should provide an interface definition using a type spec. This allows tierkreis to generate the api and use them during graph construction.
2. Rewrite or wrap the script such that it follows the tierkreis contract. In short we have to change how we read inputs and write outputs.
3. Select the script with a custom worker command in the subprocess executor configuration.


## Preliminaries

For this example we're going to use the auth worker which already has some predefined functionality.
Make sure to familiarize yourself with its contents.
In this example we're going to use a shell script with the following contents as a worker, which is contained in the `openssl_worker`.

```bash
#!/usr/bin/env bash
pk_file=$1
passphrase=$2
numbits=$3

openssl genrsa -out $pk_file -aes128 -passout "file:$passphrase" $numbits
openssl rsa -in $pk_file -passin "file:$passphrase" -pubout -out public-out
```

## Defining the interface
To do this we can define a TypeSpec dsl.

```
@portmapping
model Outputs {
    private_key: bytes
    public_key: bytes
}

interface openssl_worker {
    genrsa (
        numbits: int,
        passphrase: bytes
    ): Outputs 
}

```
It defines two outputs `private_key` and `public_key` consuming a `passphrase` and `numbits`.
The most generic type for such values is bytes, as we will be reading and writing from file directly.
Now we can generate the stubs using the cli `tkr init stubs` or from python.

In [ ]:
from pathlib import Path

from tierkreis.namespace import Namespace

if __name__ == "__main__":
    tsp_path = (
        Path().parent
        / "example_workers"
        / "openssl_worker"
        / "tkr_openssl_worker_impl"
        / "schema.tsp"
    )
    namespace = Namespace.from_spec_file(tsp_path)
    namespace.write_stubs(tsp_path.parent.parent / "api" / "api.py")

## Adapting the script to the tierkreis contract

Tierkreis writes intermediate values to storage and expects outputs to be written there as well. The subprocess executor invokes every worker with one argument: the path to a JSON-encoded `WorkerCallArgs` document. It contains the function name plus maps from input and output port names to file paths.

A shell worker can use `jq` to parse the document. Unlike the legacy shell executor, the standard protocol does not inject custom `$input_*_file` or `$output_*_file` variables.

Adding these changes to the script above yields:
```bash
#!/usr/bin/env bash
set -euo pipefail

worker_call_args=${1:?"usage: $0 WORKER_CALL_ARGS"}
error_path=$(jq -er '.error_path' "$worker_call_args")
trap 'status=$?; if (( status != 0 )); then printf "openssl_worker failed with exit code %d\n" "$status" > "$error_path"; fi' EXIT

function_name=$(jq -er '.function_name' "$worker_call_args")
if [[ $function_name != "genrsa" ]]; then
    printf 'unknown function: %s\n' "$function_name" >&2
    exit 1
fi

input_numbits=$(jq -er '.inputs.numbits' "$worker_call_args")
input_passphrase=$(jq -er '.inputs.passphrase' "$worker_call_args")
output_private_key=$(jq -er '.outputs.private_key' "$worker_call_args")
output_public_key=$(jq -er '.outputs.public_key' "$worker_call_args")
done_path=$(jq -er '.done_path' "$worker_call_args")

numbits=$(<"$input_numbits")
openssl genrsa -out "$output_private_key" -aes128 -passout "file:$input_passphrase" "$numbits"
openssl rsa -in "$output_private_key" -passin "file:$input_passphrase" -pubout -out "$output_public_key"
touch "$done_path"
```

## Building a graph using the script
Were going to build a graph that checks whether we successfully signed a message with a generated private key.


In [ ]:
from auth_worker import sign, verify
from openssl_worker import Outputs, genrsa
from tierkreis.builder import Graph
from tierkreis.models import TKR, EmptyModel


def signing_graph():
    g = Graph(EmptyModel, TKR[bool])
    message = g.const("dummymessage")
    passphrase = g.const(b"dummypassphrase")

    key_pair: Outputs = g.task(genrsa(g.const(4096), passphrase))
    private_key: TKR[bytes] = key_pair.private_key
    public_key: TKR[bytes] = key_pair.public_key

    signing_result = g.task(sign(private_key, passphrase, message)).hex_signature
    verification_result = g.task(verify(public_key, signing_result, message))
    return g.finish_with_outputs(verification_result)

which we can now run with the Rust runtime. The configuration below is the Python mapping form of `tierkreis.toml`; it uses temporary file storage so the subprocess can read and write assets.

In [ ]:
from tempfile import TemporaryDirectory

asset_directory = TemporaryDirectory(prefix="tierkreis-signing-")
worker_script = (
    Path().parent
    / "example_workers"
    / "openssl_worker"
    / "tkr_openssl_worker_impl"
    / "main.sh"
).resolve()
config = {
    "asset_storage": {
        "memory": {"type": "Memory"},
        "file": {"type": "File", "asset_dir": asset_directory.name},
    },
    "executors": {
        "memory": {"type": "Memory", "output_storage_name": "memory"},
        "subprocess": {
            "type": "Subprocess",
            "subprocess_storage_name": "file",
            "output_storage_name": "file",
            "worker_commands": {"openssl_worker": [str(worker_script)]},
        },
    },
    "runtime_state": {"type": "Memory"},
    "default_storage_name": "file",
    "default_executor_name": "subprocess",
    "logging_config": None,
}

## Configuring the subprocess executor

The subprocess executor normally discovers a worker named `openssl_worker` as `tkr-openssl-worker`. This script is not an installed package, so `worker_commands` overrides that command. In TOML, the equivalent part of the configuration is:

```toml
[executors.subprocess.worker_commands]
openssl_worker = ["/absolute/path/to/main.sh"]
```

The value is an argument vector and the runtime appends the `WorkerCallArgs` path. Other workers, such as `auth_worker` in this graph, continue to use their conventional installed commands.

In [ ]:
from tierkreis.runtime import new_from_config

runtime = await new_from_config(config)

With the runtime configured, save and run the graph.

In [ ]:
workflow = signing_graph()
with runtime:
    workflow_id = await runtime.save_workflow("openssl-signing", workflow)
    run_id = await runtime.start_new_run(workflow_id, {})
    await runtime.wait_for(run_id)
    is_verified = await runtime.get_outputs(run_id)
asset_directory.cleanup()
print(is_verified)